In [ ]:
from _client import Client
import httpx
import json

In [ ]:
base_url="https://freva.dkrz.de/api/chatbot/"

In [ ]:
url = base_url + "ping"
response=httpx.get(url=url)
print(response.json())

In [ ]:
auth_key="***REMOVED***"
input_string = "This is a test regarding your capabilities of using the code_interpreter tool and whether it supports matplotlib. Please use the code_interpreter tool to run the following code: \"import numpy as np\nimport matplotlib.pyplot as plt\nt = np.linspace(-2 * np.pi, 2 * np.pi, 100)\nsine_wave = np.sin(t)\nplt.figure(figsize=(10, 5))\nplt.plot(t, sine_wave, label='Sine Wave')\nplt.title('Sine Wave from -2π to 2π')\nplt.xlabel('Angle (radians)')\nplt.ylabel('Sine value')\nplt.axhline(0, color='black', linewidth=0.5, linestyle='--')\nplt.axvline(0, color='black', linewidth=0.5, linestyle='--')\nplt.grid()\nplt.legend()\nplt.show()\"."

In [ ]:
url = base_url + "streamresponse"

raw_response = []
with httpx.stream(method="GET", url=url, timeout=None, params={"auth_key":auth_key, "input":input_string}) as r:
    for chunk in r.iter_bytes():
        raw_response.append(chunk)

In [ ]:
def process_chunks(chunk:str, partial_response:str=""):
    chunk_split = chunk.split("}{")
    if len(chunk_split) == 1:
        if chunk[0] == "{" and chunk[-1] == "}":
            return [chunk], ""
        elif chunk[0] == "{" and chunk[-1] != "}":
            partial_response = chunk
            return [], partial_response
        elif chunk[-1] == "}":
            partial_response += chunk
            return [partial_response], ""
        else:
            partial_response += chunk
            return [], partial_response
    else:
        complete_parts = []
        for i, part in enumerate(chunk_split):
            if i==0:
                fixed_part = part + "}"
                if part[0] != "{":
                    partial_response += fixed_part
                    complete_parts.append(partial_response)
                    continue
            elif i==len(chunk_split)-1:
                fixed_part = "{" + part
                if part[-1] != "}": 
                    partial_response = fixed_part
                    return complete_parts, partial_response
                return complete_parts, ""
            else:
                fixed_part = "{" + part + "}"
            complete_parts.append(fixed_part)    

In [ ]:
complete_response = []
partial_response=""
for i, chunk in enumerate(raw_response):
    chunk_decoded=chunk.decode("utf-8")
    complete_parts, partial_response = process_chunks(chunk_decoded, partial_response)
    complete_response += complete_parts

In [ ]:
for i, r in enumerate(complete_response):
    print(i, json.loads(r))

In [ ]:
url = base_url + "streamresponse"
r=httpx.request(method="GET", url=url, timeout=None, params={"auth_key":auth_key, "input":input_string})
raw_response = r.text

In [ ]:
process_chunks(chunk=raw_response)

In [ ]:
url = base_url + "getthread"
r= httpx.get(url=url, params={"thread_id":"eMuyDSR2BWXlR4BF9jPgBO5eyjJNEMTY", "auth_key":auth_key})

In [ ]:
process_chunks(r.text)